# Influenza cross-paradigm — one Sego 2022 blueprint, three mechanical substrates

_Investigation `influenza-crossparadigm` — coder reproduction notebook._

**Question.** Can ONE backend-neutral blueprint express the whole Sego 2022 influenza model (ViralInfectionVTM) and run it faithfully on three different mechanical substrates — a Cellular Potts model (viva-cpm), a vertex model (vivarium-tyssue), and a center-based model (viva-munk) — and where do the substrates agree vs diverge when the shared mechanism is driven live?

The capstone cross-paradigm demonstration: one backend-neutral Sego 2022 influenza blueprint, compiled and run on both a Cellular Potts and a vertex substrate, with a full three-family comparison and three interactive figures.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


**Competing frameworks**

- **Cellular Potts (CPM / viva-cpm)**
- **Vertex model (tyssue / vivatyssue)**
- **Framing**


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint').is_dir():
    REPO = Path('/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_multicell_blueprint.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: `adhesion-calibration`

**Question.** Are the two substrates' mechanics defined and parameterized so that they are mechanically comparable -- per cell type and per type pair -- and which vertex parameters are fixed by the CPM, which must be calibrated, and which cannot be?

**Purpose.** CPM contact energy J (per neighbour bond) and vertex junction tension Lambda (per unit length) both penalize interface; the area constraint lambda_V (V - V_T)^2 and the area elasticity K/2 (A - A0)^2 are the same energy up to K = 2 lambda_V; the CPM temperature T drives membrane fluctuations and cell wandering that the deterministic vertex model must get from active (motility) forces. Tension contracts cells against their area elasticity, so the tension scale sets the relaxed area.

**Claim.** RE-CALIBRATED 2026-11-05 for the in-sheet immune vertex substrate (immune cells are half-size polygons in the sheet; reconnect p_4 = 1, T1 at 0.2 x side). Tension scale 1.683 (area 22.97 vs CPM 22.98). Both motilities are fold-free CAPS: epithelium 26.6 (MSD 2.2 vs CPM 9.5 um^2; fit 81), in-sheet immune 56.1 (displacement 4.6 vs 9.4 um; fit 214). Tension noise 5 (area CV 0.038 vs 0.093; 10 and 20 fold). Chemotaxis unchanged (scale 3.54, saturation 105). At kappa = 8 the small sheet now collapses completely (area 0). Earlier calibration (2026-10-28, overlay immune layer):   REVISED per the 2026-10-28 vertex-mechanics review. Both substrates' mechanics are now DEFINED term by term (docs/mechanics.md). Every vertex parameter is either energy-matched to its CPM counterpart (area stiffness K = 2 lambda_volume, per cell type), lowered per type PAIR from the blueprint adhesion table (Lambda_ab = kappa * J_ab, ratios fixed by J), or CALIBRATED to measured CPM behaviour on the Sego initial state, under a hard mesh-validity constraint (no folded faces). With per-pair junction tension actually applied and auto_reconnect ON, the tension scale IS identifiable from mean cell area: kappa = 1.74 gives area 22.8 vs CPM 23.0. Immune random-walk displacement (9.4 vs 9.4 um) and the saturating chemotactic drift curve are matched. NOT matched: epithelial wandering. The CPM-matching motility folds the vertex mesh, so it is capped at the largest fold-free value and the vertex epithelium wanders about 5x less (MSD 2.0 vs 9.5 um^2), though it does exchange neighbours (T1s). Area dispersion is not matched either (CV 0.033 vs 0.093), and the kappa that fits the small isolated sheet differs (~4).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `neutral-mechanics-sheet` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm` | 0 | backend=cpm, blueprint=workspace/blueprints/mechanics_sheet.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.sego.sego_crossparadigm`**

Sego 2022 influenza blueprint compiled onto a mechanics backend (cpm|vertex|munk): the shared field/rules/ODE/recruitment mechanism plus the one backend-specific mechanics node.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm' (Cellular Potts), 'vertex' (tyssue), or 'munk' (center-based pymunk)
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # backend-neutral blueprint: a path to a YAML file, inline YAML text, or a dict (Blueprint Studio sends inline YAML)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html (Blueprint Studio)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: adhesion-calibration ===
STUDY = 'adhesion-calibration'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_neutral_mechanics_sheet = 100
INTERVAL_neutral_mechanics_sheet = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm['state']}, core=core)
            _comp.run(STEPS_neutral_mechanics_sheet)
        print('ran', 'neutral-mechanics-sheet', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm) for', STEPS_neutral_mechanics_sheet, 'steps')
    except Exception as _e:
        print(f"neutral-mechanics-sheet not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Vertex mechanics calibration**


In [ ]:
# Vertex mechanics calibration
show_viz(_render_one('html:mechanics-calibration.html', {'title': 'Vertex knobs fitted to CPM area, wandering and chemotactic drift'}, RUNS_DB, STUDY_YAML))

**J ↔ tension sweep**


In [ ]:
# J ↔ tension sweep
show_viz(_render_one('html:adhesion-sweep.html', {'title': 'Cell-shape statistics across the vertex tension sweep vs CPM'}, RUNS_DB, STUDY_YAML))

**Neutral sheet playback — CPM vs vertex vs center-based**


In [ ]:
# Neutral sheet playback — CPM vs vertex vs center-based
show_viz(_render_one('html:neutral-sheet-playback.html', {'title': 'Mechanics-only sheet relaxation, all three paradigms (center-based not yet calibrated)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cpm-sheet-holds-size25-target _(primary)_ | kind=stat field=mean_cell_area phase=final | op in_range lo 12 hi 40 field mean_cell_area |
| vertex-sheet-comparable _(primary)_ | kind=ratio field=mean_cell_area versus=cpm | op in_range lo 0.4 hi 2.5 field area_ratio |
| tension-identifiable-from-area _(secondary)_ | kind=monotone field=mean_cell_area over=tension_scale | op strictly_decreasing note area(0.5) > area(2) > area(4); CPM area in between |
| j-tension-mapping-monotone _(secondary)_ | kind=monotone field=line_tension versus=['strength', 'scale'] | op strictly_increasing note tension(scale2) > tension(scale1); tension(strength25) > tension(strength5) |


## Study: BP-01 — the blueprint as a versioned, backend-neutral artifact (`bp-01-blueprint-schema`)

**Question.** Can the biological blueprint be specified as a versioned, backend-neutral, statically-validated artifact, independent of CPM/vertex implementations?

**Purpose.** A canonical schema + a split that routes backend/run parameters to CompileOptions makes the biological state immutable across backends; a static validator turns each semantic invariant into a checked diagnostic.

**Claim.** Stage 1 / BP-01 of the review program. The canonical blueprint is now a VERSIONED, backend-neutral artifact with a checked contract and a static validator -- the first of the three claims (blueprint fidelity) made a first-class, tested object rather than an implicit input. Specifically: (a) a `blueprint_version` field + versioned JSON schema; (b) a canonical/backend split (CompileOptions) so backend/run parameters (tension_scale, rules_seed) are kept OUT of the canonical biological state and recorded separately; (c) a static validator (`validate_blueprint` + `multicell-blueprint validate`) that reports structured diagnostics (errors AND contract warnings), not a success badge; (d) three hand-authored worked examples that validate and compile identically to BOTH backends.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `canonical-blueprint` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm` | 0 | backend=cpm, blueprint=workspace/blueprints/influenza_sego2022.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.sego.sego_crossparadigm`**

Sego 2022 influenza blueprint compiled onto a mechanics backend (cpm|vertex|munk): the shared field/rules/ODE/recruitment mechanism plus the one backend-specific mechanics node.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm' (Cellular Potts), 'vertex' (tyssue), or 'munk' (center-based pymunk)
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # backend-neutral blueprint: a path to a YAML file, inline YAML text, or a dict (Blueprint Studio sends inline YAML)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html (Blueprint Studio)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bp-01-blueprint-schema ===
STUDY = 'bp-01-blueprint-schema'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_canonical_blueprint = 100
INTERVAL_canonical_blueprint = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm['state']}, core=core)
            _comp.run(STEPS_canonical_blueprint)
        print('ran', 'canonical-blueprint', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm) for', STEPS_canonical_blueprint, 'steps')
    except Exception as _e:
        print(f"canonical-blueprint not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| same-canonical-blueprint-both-backends _(primary)_ | kind=contract field=canonical_state | op both_backends_accept note tests/test_blueprint_canonical.py::test_same_canonical_blueprint_compiles_to_both_backends |
| backend-run-params-not-canonical _(primary)_ | kind=contract field=canonical_violations | op separated_and_flagged note tests/test_blueprint_canonical.py::test_backend_run_params_are_not_canonical |
| validator-structured-diagnostics _(primary)_ | kind=contract field=validate_blueprint | op diagnostics_shape note tests/test_blueprint_canonical.py::test_validator_structured_diagnostics_and_violation_warning |


## Study: `initial-state-fidelity`

**Question.** Do both substrates faithfully reproduce the blueprint's initial centroids, sizes and types, with one vertex polygon per blueprint cell?

**Purpose.** CPM materializes each blueprint cell 1:1 at its centroid with target_volume = size. The vertex backend places the epithelial cells on a regular hexagonal lattice of the target area by an optimal 1:1 assignment (alternating with rigid recentring on the blueprint cloud). Vertex models relax well from near-hexagonal tilings; a square tiling puts every vertex at a degenerate 4-way junction. Immune cells become overlay islands at their exact centroids, and the shared fields are sampled onto the faces each step instead of being represented by polygons.

**Claim.** REVISED 2026-11-05: the vertex t0 is now ONE sheet of 48 hexagons (36 epithelial + 12 latent immune), the immune cells attached at the tissue margin site nearest their centroid instead of sitting as overlay islands on it. Epithelial centroid residual unchanged (rms 1.60, max 2.44 um); the immune cells move up to one cell diameter onto the margin (rms 4.11, max 5.60 um), closing the CPM's medium gap. All 48 cells present, types exact. Earlier reading (2026-10-28):   REVISED per the 2026-10-28 vertex-mechanics review. Both backends reproduce the blueprint's t=0 state (every cell's centroid, size and type), with every vertex polygon corresponding to exactly ONE Sego cell. The CPM side is EXACT (centroid residual 0). The vertex epithelium is a regular HEXAGONAL sheet at the target area (36 hexagons, area 25.0 each) mapped 1:1 onto the blueprint cells (sub-cell residual: RMS 1.60 um, max 2.44 um). The 12 immune cells are overlay islands exactly on their centroids, and the fields are overlaid on the tissue rather than meshed: no guard-ring or medium polygons remain.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sego-t0-fidelity` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm` | 0 | backend=cpm, blueprint=workspace/blueprints/influenza_sego2022.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.sego.sego_crossparadigm`**

Sego 2022 influenza blueprint compiled onto a mechanics backend (cpm|vertex|munk): the shared field/rules/ODE/recruitment mechanism plus the one backend-specific mechanics node.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm' (Cellular Potts), 'vertex' (tyssue), or 'munk' (center-based pymunk)
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # backend-neutral blueprint: a path to a YAML file, inline YAML text, or a dict (Blueprint Studio sends inline YAML)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html (Blueprint Studio)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: initial-state-fidelity ===
STUDY = 'initial-state-fidelity'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sego_t0_fidelity = 100
INTERVAL_sego_t0_fidelity = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm['state']}, core=core)
            _comp.run(STEPS_sego_t0_fidelity)
        print('ran', 'sego-t0-fidelity', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm) for', STEPS_sego_t0_fidelity, 'steps')
    except Exception as _e:
        print(f"sego-t0-fidelity not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**t0 centroid residuals**


In [ ]:
# t0 centroid residuals
show_viz(_render_one('html:t0-residuals.html', {'title': 'Per-cell t0 centroid residual, CPM vs vertex'}, RUNS_DB, STUDY_YAML))

**t0 side-by-side**


In [ ]:
# t0 side-by-side
show_viz(_render_one('html:t0-side-by-side.html', {'title': 'Initial state: CPM | vertex'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cpm-exact-t0 _(primary)_ | kind=residual field=centroid_residual backend=cpm | op lt value 1e-06 field centroid_rms |
| vertex-clean-identity _(primary)_ | kind=residual field=centroid_residual backend=vertex | op lt value 2.0 field centroid_rms note fraction_clean == 1.0; n_guard_faces == 0 |
| vertex-one-hexagonal-sheet _(primary)_ | kind=topology field=vertex_residual_report | op eq value 36 field n_sheet note all sheet faces 6-sided, area 25, one component; n_overlay == 12 |
| all-types-match _(primary)_ | kind=count field=type_match | op eq value 48 field n_type_match |


## Study: `flu-lung-crossparadigm`

**Question.** Does one backend-neutral blueprint reproduce the Sego 2022 influenza model on two different mechanical substrates, and -- judged against same-backend seed noise -- do the substrates diverge?

**Purpose.** The epithelial rates, four reaction-diffusion fields, Price-2015 systemic ODE (fed live by CellCountStep) and latent-pool immune recruitment are compiled identically onto both backends; only the single mechanics node differs (CPM Metropolis flips vs tyssue vertex relaxation). If the substrate mattered beyond noise, the cross-backend metrics would fall OUTSIDE the same-backend seed spread.

**Claim.** REVISED 2026-11-05 (reviewer feedback: vertex CD8/NK floated free of the tissue): vertex immune cells are now SMALLER POLYGONS IN THE EPITHELIAL SHEET, attached at the tissue margin and moving along it by neighbour exchange, as the CPM immune pool wets the epithelial edge; reconnect resolves merged rosettes faster (p_4 = 1, T1 at 0.2 x side); motilities re-calibrated under the no-fold cap. On this substrate FATE still falls inside the same-backend seed spread (cross 0.948 vs 0.926-0.960), but infected-patch OCCUPANCY (IoU 0.053, below both baselines 0.100 / 0.299) and LOCATION (centroid distance 5.7 vs 3.3 um) now separate the backends -- a candidate substrate signal, not yet causally attributed. Earlier reading (2026-10-28, overlay immune layer) kept for provenance: The ONE backend-neutral Sego 2022 influenza blueprint compiles to BOTH the CPM (viva-cpm) and vertex (vivarium-tyssue) substrates and runs the Sego model WITH NAMED DEVIATIONS (V/L/C/F producers absent, latent-pool recruitment approximation) end-to-end on each. RE-RUN 2026-10-28 on the REVISED vertex substrate: a hexagonal one-polygon-per-cell sheet, immune overlay, auto_reconnect, per-pair tension, domain walls, and CPM-calibrated motility under a no-fold constraint. Vertex cells now move within the tissue and exchange neighbours, though the epithelium wanders ~5x less than on the CPM. Replicate runs are now also stochastic in their MECHANICS, so the same-backend null control includes mechanical noise. Against that null, cell FATE, infected-patch OCCUPANCY and patch LOCATION all fall inside the same-backend seed spread (fail to reject). The one candidate substrate signal left: the vertex I-count leaves the narrow CPM 3-seed envelope for ~45% of steps by N=400.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sego-influenza-crossparadigm` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm` | 0 | backend=cpm, blueprint=workspace/blueprints/influenza_sego2022.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.sego.sego_crossparadigm`**

Sego 2022 influenza blueprint compiled onto a mechanics backend (cpm|vertex|munk): the shared field/rules/ODE/recruitment mechanism plus the one backend-specific mechanics node.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm' (Cellular Potts), 'vertex' (tyssue), or 'munk' (center-based pymunk)
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # backend-neutral blueprint: a path to a YAML file, inline YAML text, or a dict (Blueprint Studio sends inline YAML)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html (Blueprint Studio)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: flu-lung-crossparadigm ===
STUDY = 'flu-lung-crossparadigm'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sego_influenza_crossparadigm = 100
INTERVAL_sego_influenza_crossparadigm = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm['state']}, core=core)
            _comp.run(STEPS_sego_influenza_crossparadigm)
        print('ran', 'sego-influenza-crossparadigm', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm) for', STEPS_sego_influenza_crossparadigm, 'steps')
    except Exception as _e:
        print(f"sego-influenza-crossparadigm not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial playback — CPM vs vertex**


In [ ]:
# Spatial playback — CPM vs vertex
show_viz(_render_one('html:spatial-playback.html', {'title': 'Spatial playback: CPM | vertex'}, RUNS_DB, STUDY_YAML))

**Dynamics overlay — CPM vs vertex (I-count band)**


In [ ]:
# Dynamics overlay — CPM vs vertex (I-count band)
show_viz(_render_one('html:dynamics-overlay.html', {'title': 'Dynamics vs the CPM stochastic-envelope band'}, RUNS_DB, STUDY_YAML))

**Divergence map**


In [ ]:
# Divergence map
show_viz(_render_one('html:divergence.html', {'title': 'Per-cell agreement + spatial map (vs same-backend baseline)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| both-backends-run-end-to-end _(primary)_ | kind=count field=frames when=end-of-run | op eq value 151 field n_frames_at_N150 |
| fate-agreement-within-seed-baseline _(primary)_ | kind=delta field=fate_agreement compare_to=same_backend_cpm_seed_baseline | op gte value -0.05 field cross_minus_baseline |
| spatial-not-representation-confounded _(primary)_ | kind=ratio field=final_mean_area versus=cpm | op in_range lo 0.6 hi 1.6 field area_ratio |
| vertex-within-cpm-envelope _(secondary)_ | kind=fraction field=inband | op gte value 0.8 field vertex_inband_fraction |
| three-figures-render-under-cap _(secondary)_ | kind=size field=figure_html_bytes | op lt value 2500000 field figure_html_bytes |


## Study: `dose-response-crossparadigm`

**Question.** Do the two substrates agree on the SIGN of the dose response over initial viral load (more load -> more infection), robustly across seeds?

**Purpose.** More initially-infected cells secrete more virus into the shared field, raising the per-cell infection probability (Pr = 1 - exp(-g_hv * virus)) everywhere, so the final infected count rises with the initial dose on either substrate.

**Claim.** RE-RUN 2026-11-05 on the in-sheet immune vertex substrate (re-calibrated motility): figure regenerated; the STRICT-DIRECTION behavior test (highest dose ends with more infection than the lowest, on both backends) still passes. Earlier reading:   Both backends agree on the DIRECTION of the dose response: a HIGHER initial viral load (more initially-infected epithelial cells) ends with STRICTLY more infection than a LOWER load, on every backend and seed tested. This is a directional (lowest-vs-highest) result; intermediate-dose monotonicity is seed-dependent and is NOT claimed.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sego-dose-sweep` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm` | 0 | backend=cpm, blueprint=workspace/blueprints/influenza_sego2022.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.sego.sego_crossparadigm`**

Sego 2022 influenza blueprint compiled onto a mechanics backend (cpm|vertex|munk): the shared field/rules/ODE/recruitment mechanism plus the one backend-specific mechanics node.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm' (Cellular Potts), 'vertex' (tyssue), or 'munk' (center-based pymunk)
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # backend-neutral blueprint: a path to a YAML file, inline YAML text, or a dict (Blueprint Studio sends inline YAML)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html (Blueprint Studio)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: dose-response-crossparadigm ===
STUDY = 'dose-response-crossparadigm'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sego_dose_sweep = 100
INTERVAL_sego_dose_sweep = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm['state']}, core=core)
            _comp.run(STEPS_sego_dose_sweep)
        print('ran', 'sego-dose-sweep', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm) for', STEPS_sego_dose_sweep, 'steps')
    except Exception as _e:
        print(f"sego-dose-sweep not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Dose response — initial load → final infection**


In [ ]:
# Dose response — initial load → final infection
show_viz(_render_one('html:dose-response.html', {'title': 'Final infected count vs initial dose, both backends and seeds'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| strict-direction-both-backends-and-seeds _(primary)_ | kind=comparison field=final_infected_count low=0.03 high=0.15 | op strictly_greater note final(0.15) > final(0.03) for cpm/vertex/munk x seeds {0,1} |
| higher-dose-seeds-more-cells _(primary)_ | kind=comparison field=seed_infected low=0.03 high=0.15 | op strictly_greater value 5 > 1 |


## Study: Backend equivalence (TOST) (`crossparadigm-equivalence`)

**Question.** Are CPM and vertex EQUIVALENT (not merely "not-distinguishable") under replication -- and on which outcome: cell fate, exact spatial position, or both?

**Purpose.** Fate is driven by the SHARED field/rule/ODE mechanism (identical across backends), so cell-fate outcomes should be backend-neutral. Exact cell position is set by each backend's own mechanics (CPM lattice Metropolis vs vertex tension relaxation), which differ, so positions may drift apart even when fate agrees.

**Claim.** RE-RUN 2026-11-05 on the in-sheet immune vertex substrate (immune cells are smaller polygons attached to the tissue margin; faster reconnect; re-calibrated motility). Verdicts unchanged: FATE equivalent (Delta -0.0014, cluster CI [-0.0034, +0.0008] inside +/-0.05); exact POSITION not equivalent (same 0.674, cross 0.547, Delta +0.126, cluster CI [+0.070, +0.181] vs +/-0.10). Diagnostic split: the gap is not only the immune margin attachment (t0 offset up to 5.6 um) -- the epithelium alone is also not equivalent (same 0.738, cross 0.622, Delta +0.116); immune-only Delta +0.156. Earlier reading (2026-10-28) kept for provenance: SCOPED to Claim 3 (scientific equivalence) -- NOT blueprint fidelity, NOT biological validity. RE-RUN 2026-10-28 on the revised vertex substrate (hexagonal sheet, immune overlay, auto_reconnect, per-pair tension, CPM-calibrated motility) with mechanics-stochastic replicates. On cell FATE the substrates remain equivalent under the dependence-aware run-cluster bootstrap: Delta(same-cross) -0.0014, 90% CI [-0.0034, +0.0008], inside +/-0.05 (margin not yet biologically justified). On exact POSITION they are still NOT equivalent, but the gap shrank about 7-fold: cross 0.472 vs same-backend 0.608 (Delta +0.135, cluster CI [+0.071, +0.210] vs margin +/-0.10). The previous spatial gap (cross 0.083 vs same 1.000) mostly reflected a degenerate null. Replicates differed only in rule draws, so same-backend positions were identical by construction, and the vertex cells barely moved.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sego-equivalence-grid` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm` | 0 | backend=cpm, blueprint=workspace/blueprints/influenza_sego2022.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.sego.sego_crossparadigm`**

Sego 2022 influenza blueprint compiled onto a mechanics backend (cpm|vertex|munk): the shared field/rules/ODE/recruitment mechanism plus the one backend-specific mechanics node.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm' (Cellular Potts), 'vertex' (tyssue), or 'munk' (center-based pymunk)
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # backend-neutral blueprint: a path to a YAML file, inline YAML text, or a dict (Blueprint Studio sends inline YAML)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html (Blueprint Studio)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: crossparadigm-equivalence ===
STUDY = 'crossparadigm-equivalence'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sego_equivalence_grid = 100
INTERVAL_sego_equivalence_grid = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm['state']}, core=core)
            _comp.run(STEPS_sego_equivalence_grid)
        print('ran', 'sego-equivalence-grid', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm) for', STEPS_sego_equivalence_grid, 'steps')
    except Exception as _e:
        print(f"sego-equivalence-grid not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Fate equivalence (TOST)**


In [ ]:
# Fate equivalence (TOST)
show_viz(_render_one('html:equivalence-fate.html', {'title': 'Cross- vs same-backend fate agreement with the equivalence margin'}, RUNS_DB, STUDY_YAML))

**Spatial (exact-position) equivalence (TOST)**


In [ ]:
# Spatial (exact-position) equivalence (TOST)
show_viz(_render_one('html:equivalence-spatial.html', {'title': 'Cross- vs same-backend exact-position agreement'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| fate-equivalent-tost _(primary)_ | kind=equivalence field=fate_agreement margin=0.05 | op tost_equivalent note 90% CI of (same-cross) inside +/-0.05 |
| spatial-gap-is-dynamic-not-offset _(secondary)_ | kind=comparison field=spatial_agreement low=0.0 high=1.0 | op strictly_greater note cross spatial(t0) > cross spatial(final) |


## Study: Mechanics-dominated counterfactual (positive control) (`substrate-divergence-counterfactual`)

**Question.** When MECHANICS (not the shared mechanism) drives the outcome, does the equivalence test correctly REJECT -- i.e. does it have the power to detect a real substrate difference?

**Purpose.** CPM moves cells by Metropolis acceptance of lattice copies (chemotaxis as an energy bias); the vertex fork moves them by the ActiveMigration director sampled from the field gradient. Under a strong gradient these displace cells by different amounts, and CPM's Potts neighbour-swapping sorts differentially- adhesive cells in a way the vertex topology resists -- so the spatial outcome genuinely depends on the substrate.

**Claim.** RE-RUN 2026-11-05 (reviewer: "auto-reconnect isn't working here"). Reconnect WAS on; it is now also faster (merged 4-way vertices detach at p_4 = 1 per unit time instead of 0.1, T1 at 0.2 x side), and the playback's vertex panel counts neighbour exchanges frame by frame. In this regime the vertex run still makes only 3 exchanges in 80 steps (vs 74 in the Sego run): with 15 A cells (nearly all on the border) and a one-row B stripe, nothing shrinks a junction to the T1 threshold, so the vertex clusters keep their topology while the CPM blobs flow -- the substrate difference this control is built to expose, not a disabled reconnect. Equivalence is still REJECTED (same 0.775, cross 0.450, Delta +0.325 vs +/-0.10). Earlier reading (2026-10-28): In a MECHANICS-DOMINATED regime (strong differential adhesion + a strong chemoattractant gradient, no shared rules/ODE), the spatial equivalence test still REJECTS equivalence after the 2026-10-28 vertex revision. Cross-backend spatial agreement is 0.417 vs same-backend 0.775 (diff +0.358 > margin 0.10), now with mechanics-stochastic replicates, so the same-backend baseline is a genuine stochastic null rather than 1.0 by construction. It remains the POSITIVE CONTROL: the same test that finds the Sego model fate-equivalent detects a substrate difference here. REVISED MECHANISM: the earlier attribution (vertex A cells climbing the gradient to y~17.9 while CPM stays at ~10.7) was an artifact of the old uncalibrated vertex chemotaxis. With chemotactic drift calibrated to the CPM (saturating), neither substrate moves A far (CPM mean y 11.0 -> 11.0, vertex 10.7 -> 10.1). The divergence is now positional / sorting dynamics, not migration direction, and is not yet causally isolated.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `mechanics-divergence` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm` | 0 | backend=cpm, blueprint=workspace/blueprints/mechanics_divergence.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.sego.sego_crossparadigm`**

Sego 2022 influenza blueprint compiled onto a mechanics backend (cpm|vertex|munk): the shared field/rules/ODE/recruitment mechanism plus the one backend-specific mechanics node.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm' (Cellular Potts), 'vertex' (tyssue), or 'munk' (center-based pymunk)
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # backend-neutral blueprint: a path to a YAML file, inline YAML text, or a dict (Blueprint Studio sends inline YAML)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html (Blueprint Studio)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: substrate-divergence-counterfactual ===
STUDY = 'substrate-divergence-counterfactual'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_mechanics_divergence = 100
INTERVAL_mechanics_divergence = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm['state']}, core=core)
            _comp.run(STEPS_mechanics_divergence)
        print('ran', 'mechanics-divergence', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm) for', STEPS_mechanics_divergence, 'steps')
    except Exception as _e:
        print(f"mechanics-divergence not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial equivalence — REJECTED (TOST)**


In [ ]:
# Spatial equivalence — REJECTED (TOST)
show_viz(_render_one('html:equivalence-spatial.html', {'title': 'Cross- vs same-backend spatial agreement in the divergent regime'}, RUNS_DB, STUDY_YAML))

**Mechanics-dominated playback — CPM vs vertex**


In [ ]:
# Mechanics-dominated playback — CPM vs vertex
show_viz(_render_one('html:spatial-playback.html', {'title': 'Divergent migration/sorting: CPM voxel | vertex polygon'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| spatial-equivalence-rejected _(primary)_ | kind=equivalence field=spatial_agreement margin=0.1 | op tost_not_equivalent note cross spatial agreement << same; diff > margin |
| migration-direction-differs _(primary)_ | kind=comparison field=typeA_mean_y_change low=-40.0 high=40.0 | op lt value 3.0 field abs_typeA_mean_y_change note both backends |


## Study: Three substrates on the large Sego model (behavior-metric distances) (`large-crossparadigm-comparison`)

**Question.** On the LARGE Sego 2022 blueprint (120x120, ~225-cell epithelium), how far does each mechanical substrate's emergent MOVEMENT and INTERACTION pattern sit from the CPM reference, measured by the cross-paradigm behavior metrics (per-cell displacement, MSD, neighbour-graph churn, infection-front speed, cell sorting)? Each metric is a DISTANCE from the CPM; the same-backend seed-to-seed distance is the null floor a match must approach.

**Purpose.** All three backends run the SAME shared mechanism (fields, rules, immune ODE) on the SAME large blueprint; only the mechanics node differs. Differences in the behavior metrics are therefore attributable to the substrate (CPM Metropolis lattice vs vertex confluent polygons vs center-based discs), not the biology.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sego-large` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm_large` | 0 | backend=cpm |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.sego.sego_crossparadigm_large`**

Scaled-up Sego 2022 influenza preset: a biology-preserving ~2.4x linear scale-up of 'sego_crossparadigm' (120x120 patch, 15x15 ~225-cell epithelial block) -- a bigger tissue, the SAME field/rules/ODE/recruitment mechanism. A heavier starting point to run from in the Blueprint Studio.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm_large = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm' (Cellular Potts), 'vertex' (tyssue), or 'munk' (center-based pymunk)
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022_large.yaml',   # backend-neutral blueprint: a path to a YAML file, inline YAML text, or a dict (defaults to the scaled-up Sego preset)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html (Blueprint Studio)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: large-crossparadigm-comparison ===
STUDY = 'large-crossparadigm-comparison'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sego_large = 100
INTERVAL_sego_large = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm_large if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm_large' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm_large = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm_large', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm_large['state']}, core=core)
            _comp.run(STEPS_sego_large)
        print('ran', 'sego-large', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm_large) for', STEPS_sego_large, 'steps')
    except Exception as _e:
        print(f"sego-large not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| munk-moves-and-interacts _(primary)_ | kind=distance field=per_cell_displacement_distance backend=munk | op gt value 0.5 note munk displacement distance >> 0 (was ~0 when frozen) |
| vertex-closest-to-cpm _(primary)_ | kind=comparison field=msd_distance | op lt note vertex_vs_cpm < munk_vs_cpm on displacement and MSD |
| distances-above-null-floor _(primary)_ | kind=null_control field=all_metrics | op reported note cross vs null floor reported per metric |


## Study: Vertex → CPM fluctuation match (`vertex-cpm-fluctuation-match`)

**Question.** Can the vertex (tyssue) backend reproduce the CPM's emergent interaction pattern on the large Sego 2022 blueprint — its per-cell motility, boundary fluctuation and neighbour-exchange statistics — WITHOUT folding the mesh? Today the honest CPM-matching motility/noise values invert short junctions and self-intersect the polygons, so they were capped, leaving the vertex epithelium ~5x less mobile and far more ordered than the CPM. This study asks whether a mesh-safe mechanism closes that gap under a quantitative distance-to-CPM target.

**Purpose.** CPM advances a stochastically-roughened voxel interface by Metropolis boundary flips at temperature; the vertex fork advances a confluent, gap-free polygon tiling by grad-E displacement plus discrete T1/T2/T3 neighbour swaps. The CPM's per-voxel fluctuation has no direct vertex analog — it is approximated by OU junction-tension noise plus an active-Brownian random walk, both of which fold the mesh at CPM-matching amplitude and were therefore capped (adhesion-calibration).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `cpm-reference` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm` | 0 | backend=cpm |
| `vertex-shipped` | `viva_multicell_blueprint.composites.sego.sego_crossparadigm` | 0 | backend=vertex |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.sego.sego_crossparadigm`**

Sego 2022 influenza blueprint compiled onto a mechanics backend (cpm|vertex|munk): the shared field/rules/ODE/recruitment mechanism plus the one backend-specific mechanics node.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm' (Cellular Potts), 'vertex' (tyssue), or 'munk' (center-based pymunk)
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # backend-neutral blueprint: a path to a YAML file, inline YAML text, or a dict (Blueprint Studio sends inline YAML)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html (Blueprint Studio)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: vertex-cpm-fluctuation-match ===
STUDY = 'vertex-cpm-fluctuation-match'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_cpm_reference = 100
INTERVAL_cpm_reference = 1.0
STEPS_vertex_shipped = 100
INTERVAL_vertex_shipped = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm['state']}, core=core)
            _comp.run(STEPS_cpm_reference)
        print('ran', 'cpm-reference', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm) for', STEPS_cpm_reference, 'steps')
    except Exception as _e:
        print(f"cpm-reference not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm if 'ov_viva_multicell_blueprint_composites_sego_sego_crossparadigm' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm = materialize_composite('viva_multicell_blueprint.composites.sego.sego_crossparadigm', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_sego_sego_crossparadigm['state']}, core=core)
            _comp.run(STEPS_vertex_shipped)
        print('ran', 'vertex-shipped', f'(viva_multicell_blueprint.composites.sego.sego_crossparadigm) for', STEPS_vertex_shipped, 'steps')
    except Exception as _e:
        print(f"vertex-shipped not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| epithelial-motility-matches-cpm _(primary)_ | kind=distance field=epithelial_msd reference=cpm-reference | op within_fraction value 0.3 note |MSD_vertex - MSD_cpm| / MSD_cpm <= 0.30 |
| boundary-fluctuation-matches-cpm _(primary)_ | kind=distance field=area_cv reference=cpm-reference | op within_fraction value 0.3 |
| interaction-pattern-distance-drops _(primary)_ | kind=distance field=neighbour_graph_churn reference=cpm-reference | op lt_baseline note candidate distance < vertex-shipped distance |
| mesh-stays-valid _(gate)_ | kind=invariant field=mesh_valid | op all_true note mesh validity holds every step |


## Open decisions
- STAGE 1 (BP-01): formalize a versioned blueprint schema + canonical IR, separate from backend geometric init (units, coord frame, boundaries, tessellation-independent IDs, footprints, fields, contacts, latent states, seeds, init order, mechanism bindings, provenance, allowable loss).
- STAGE 2 (BP-02/BP-04, extend initial-state-fidelity): prove blueprint fidelity beyond centroids/IDs — area, footprint, contact, adjacency, boundary, topology, round-trip, with explicit error measures and loss reports.
- STAGE 3 (BP-07): adhesion-calibration now calibrates the vertex tension scale, immune motility and chemotaxis to CPM observables under a no-fold constraint (2026-10-28). Open: the vertex epithelium cannot reach CPM fluidity without folding (MSD 2.0 vs 9.5 um^2) -- needs a fold-aware topology handler or a softer epithelium; area dispersion unmatched; tension scale geometry-dependent (1.74 vs ~4); one cell size.
- STAGE 4 (BP-05): V/L/C/F inputs have no producer and immune recruitment is a latent-pool approximation — build a producer/consumer ledger, implement or clearly designate deviations, and ablate frozen inputs.
- STAGE 4 (BP-06): the ±0.05 fate-equivalence margin is NOT yet biologically justified; the dependence-aware reanalysis is done (cluster bootstrap) but power under biologically meaningful perturbations is not established.
- STAGE 4 (BP-08): biological fidelity to Sego 2022 is NOT validated — digitize figure/table/parameter anchors and compare predictive intervals on held-out data; no biology inferred from cross-backend agreement.
- STAGE 4 (dose-response redefine): separate virion inoculum from seeded infected fraction; 6-10 graded doses x replicate seeds; it is currently directional-only, NOT a dose-response curve.
- STAGE 5 (BP-09): compile a SECOND independent tissue blueprint through the same schema with no influenza-specific adaptations — the decisive generality test.
